In [0]:
# 05_quality_checks – automatische Tests; schlägt der Job fehl, stimmt etwas nicht
from pyspark.sql import functions as F

DB = "workspace.banking"
tx = spark.table(f"{DB}.silver_transactions")

checks = {
    "Silver nicht leer":              tx.count() > 0,
    "transaction_id eindeutig":       tx.count() == tx.select("transaction_id").distinct().count(),
    "keine NULL-Beträge":             tx.filter(F.col("amount_eur").isNull()).count() == 0,
    "nur EUR/USD/CHF":                tx.filter(~F.col("currency").isin("EUR", "USD", "CHF")).count() == 0,
    "Bronze (eindeutig) = Silver + Quarantäne":
        spark.table(f"{DB}.bronze_transactions").select("transaction_id").distinct().count()
        == tx.count() + spark.table(f"{DB}.quarantine_transactions").count(),
}
for name, ok in checks.items():
    print(("OK    " if ok else "FEHLER"), name)
assert all(checks.values()), "Mindestens ein Qualitätscheck ist fehlgeschlagen!"